In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
# set folders
try:
    onedrive_folder = os.environ['OneDriveCommercial'].replace('\\','/') + '/'
except:
    onedrive_folder = os.environ['OneDrive'].replace('\\','/') + '/'
print(onedrive_folder)

data_folder = 'Documents/projects/frailty/ML analysis/data/'

# set files
data_file = 'cleaned_data_2026-06-01_unktags_2014.csv'

### Define utility functions

In [ ]:
from sklearn.utils import resample
import statsmodels.api as sm
from scipy.special import logit

def bootstrap_calibration_metrics(y_test, y_pred_proba, n_bootstrap=500, random_state=123):
    """
    Calculate calibration metrics with bootstrap confidence intervals
    
    Parameters:
    -----------
    y_test : array-like, true binary labels (0/1)
    y_pred_proba : array-like, predicted probabilities
    algorithm: name of algorithm which produced these probabilities
    n_bootstrap : int, number of bootstrap iterations
    random_state : int, random seed for reproducibility
    
    Returns:
    --------
    dict: calibration metrics with 95% CIs from bootstrap
    """
    np.random.seed(random_state)

    # convert y_test from Series to array, otherwise when you try to index it with indices (which are integer positions), Pandas will try to use those integers as index labels rather than positional indices.
    y_test = y_test.values

    # Handle extreme probabilities to avoid logit issues
    eps = 1e-10
    y_pred_proba_clipped = np.clip(y_pred_proba, eps, 1 - eps)
    # The logit function (log-odds transformation) is used here. when p=0, logit(p)=neg infinity. when p=1, logit(p)=pos infinity. Infinity breaks calculations! The test can't compute with infinite values. using np.clip forces all values to be between lower and upper bounds
    # 1. The clipping is EXTREMELY tiny (1e-10 = 0.0000000001)
    #    This is practically indistinguishable from 0 or 1 for any real application
    # 2. It only affects predictions that are exactly 0 or exactly 1
    #    Most models never predict exactly 0 or 1 anyway
    # 3. It prevents mathematical errors without changing results meaningfully
    
    # 2. Calibration slope using logistic regression
    # Transform to log-odds
    logit_pred = logit(y_pred_proba_clipped)
    
    # Add constant for intercept
    X = sm.add_constant(logit_pred)
    
    # Fit logistic regression
    try:
        model = sm.Logit(y_test, X)
        result = model.fit(disp=0)

        cal_in_large = result.params[0]   # intercept
        slope = result.params[1]          # slope
        
    except Exception as e:
        print(f"Warning: Logistic regression failed: {e}")
        cal_in_large = np.nan
        slope = np.nan
   
    # Initiate list to store bootstrap results
    bootstrap_intercepts = []
    bootstrap_slopes = []

    n = len(y_test)
    
    # Run bootstrap iterations
    for i in range(n_bootstrap):
        # Bootstrap sample with replacement
        indices = resample(range(n), n_samples=n, replace=True)
        y_test_boot = y_test[indices]
        y_proba_boot = y_pred_proba_clipped[indices]
        
        # Calculate metrics on bootstrap sample
        try:            
            # Calibration slope
            logit_pred_boot = logit(y_proba_boot)
            X_boot = sm.add_constant(logit_pred_boot)
            
            model_boot = sm.Logit(y_test_boot, X_boot)
            result_boot = model_boot.fit(disp=0)

            bootstrap_intercepts.append(result_boot.params[0])
            bootstrap_slopes.append(result_boot.params[1])
            
        except Exception as e:
            print(f"Warning: Bootstrap failed: {e}")
            # Skip failed iterations (rare)
            continue
    
    # Calculate bootstrap confidence intervals (2.5th and 97.5th percentiles)
    cal_in_large_ci = np.percentile(bootstrap_intercepts, [2.5, 97.5])
    slope_ci = np.percentile(bootstrap_slopes, [2.5, 97.5])

    # produce output
    final_citl = f"{cal_in_large:.3f} ({cal_in_large_ci[0]:.3f} - {cal_in_large_ci[1]:.3f})"
    final_slope = f"{slope:.3f} ({slope_ci[0]:.3f} - {slope_ci[1]:.3f})"

    return (final_citl, final_slope)

In [ ]:
# load in data
slsr10 = pd.read_csv(onedrive_folder+data_folder+data_file, low_memory=False)
slsr10.info()

In [ ]:
# remove the 'ac' variables on brain location
ac_vars = [var for var in slsr10.columns if var.startswith('ac')]
print(ac_vars)
slsr10 = slsr10.drop(columns=ac_vars)
slsr10.info()

In [ ]:
slsr10.describe(include='all')

In [ ]:
# set aside some data for temporal validation

test_df = slsr10[slsr10['strk_y']>2022]
print(len(test_df))

train_df = slsr10[slsr10['strk_y']<=2022]
print(len(train_df))

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# set a seed for reproducibility
seed = 123

# Store SHAP values
shap_dict = {}

# initiate empty dict to store models pred and proba for delong and mcnemar tests
models_pred = {}
models_proba = {}

# initiate empty list to store all results
results_table = []
results_graph = []
threshold_list = []

# Identify column types
numerical_cols = ['age', 'onset_to_presentation']               # Continuous
ordinal_cols = ['batot', 'imd_decile', 'glas_cs', 'strk_y']            # Ordinal integers
cols_to_scale = numerical_cols + ordinal_cols

# Prepare data
cols_to_drop = ['id','bi0.25','bi1','death1','death0.25','efi_score'] + [col for col in slsr10.columns if col.startswith('ac')]
print(f'cols to drop: {cols_to_drop}')
X_train = train_df.drop(columns=cols_to_drop, axis=1)
y_train = train_df['death1']

X_test = test_df.drop(columns=cols_to_drop, axis=1)
y_test = test_df['death1']

# Create ColumnTransformer that ONLY scales specified columns
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), cols_to_scale)
    ],
    remainder='passthrough'  # Everything else passes through untouched
)

# Fit and transform
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

# reconstruct column names in the scaled dataset because ColumnTransformer gives a pure numpy array, and pandas column names are lost in the process.
all_columns = preprocessor.get_feature_names_out()
# Convert to DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=all_columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=all_columns, index=X_test.index)
# index=X_train.index: Maintains row identity

# inspect
print(f'Training dataset, before scaling: {X_train.shape}')
print(f'Training dataset, after scaling: {X_train_scaled.shape}')
print(f'Test dataset, before scaling: {X_test.shape}')
print(f'Test dataset, after scaling: {X_test_scaled.shape}')

In [ ]:
# event rate
print(f'Out of {len(y_train)} training samples, we have {y_train.sum()} events, or event rate of {y_train.mean()}')
print(f'Out of {len(y_test)} test samples, we have {y_test.sum()} events, or event rate of {y_test.mean()}')

### Logistic Regression

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, precision_recall_curve, average_precision_score, log_loss
from sklearn.calibration import calibration_curve
import shap

# Train
model = LogisticRegression(max_iter=10000, random_state=seed) # implicitly by default minimizing log loss. cannot change this setting.
model.fit(X_train_scaled, y_train)

# get feature importance
explainer = shap.LinearExplainer(model, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)
    # Linear Explainer takes the model object and directly extracts the coefficient and intercept. It requires a second argument X_train_scaled because it will need to compute the average/baseline for the group.
    # then we call .shap_values on this instance of the Explainer. for each patient Pi and each feature Fi, it computes SHAP(Pi) = coef(Fi) * [Pi's individual value for this feature - group average]
    # in the end, we get a matrix of (n_patients, n_features) SHAP values
# store into dict
shap_dict['Logistic Regression'] = shap_values

# Tune the decision threshold on the best model
# get probabilities for 30 different CV runs of the same (best) model
y_proba_thresh = cross_val_predict(
    model, X_train_scaled, y_train, 
    method='predict_proba', cv=StratifiedKFold(30, shuffle=True, random_state=seed)
)[:, 1] 

# Threshold tuning
# use the 30 probabilities to compute a bunch of F1 scores
# identify the best f1 score and designate it as the best cutoff
precision, recall, thresh = precision_recall_curve(y_train, y_proba_thresh)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9) 
optimal_threshold = thresh[np.argmax(f1_scores)] # argmax() gives the index position, not the scalar f1 value, of the highest f1 score

# store into list
for t, f1 in zip(thresh, f1_scores):
    threshold_list_row = {
        'algorithm': 'Logistic Regression',
        'threshold': t,
        'f1_scores': f1
    }
    threshold_list.append(threshold_list_row)

# EVALUATE ON TEST SET
# Predict
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]  # For ROC-AUC, we get probabilities of class 1
y_pred = (y_pred_proba >= optimal_threshold).astype(int)  # Class predictions (0/1)

# Calculate metrics
test_log_loss = log_loss(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba) # for graph plotting ROC curve
# Better approach: Precision-Recall Curve
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba) 
# Get data for calibration analyses
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=5, strategy='uniform')
final_citl, final_slope = bootstrap_calibration_metrics(y_test, y_pred_proba)
    
print(f"Test Log Loss: {test_log_loss:.3f}")
print(f"Test F1: {f1:.3f}")
print(f"Optimal Threshold: {optimal_threshold:.3f}")
print(f"Test ROC-AUC: {roc_auc:.3f}")
print(f"Test PR-AUC: {pr_auc:.3f}")
print(f"Test Calibration CITL: {final_citl}")
print(f"Test Calibration Slope: {final_slope}")

# Append to table
results_table.append({
    'algorithm': 'Logistic Regression',
    'log_loss': test_log_loss,
    'f1': f1,
    'optimal_threshold': optimal_threshold,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'final_citl': final_citl,
    'final_slope': final_slope
})

# Append to storage list for plotting graph
results_graph.append({
    'algorithm': 'Logistic Regression',
    'roc_auc': roc_auc,
    'fpr': fpr,
    'tpr': tpr,
    'pr_auc': pr_auc,
    'precision': precision,
    'recall': recall,
    'prob_true': prob_true,
    'prob_pred': prob_pred
})

# Append to pred and proba dicts for running delong and mcnemar tests
models_pred['Logistic Regression'] = y_pred
models_proba['Logistic Regression'] = y_pred_proba

In [ ]:
plt.hist(y_pred_proba, bins=15)
plt.title("LR predicted probabilities")

In [ ]:
# ===========================
# Learning Curve for Log Loss
# ===========================

from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np

# Generate learning curve with log loss
train_sizes, train_scores, val_scores = learning_curve(
    model, 
    X_train_scaled, 
    y_train,
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
    scoring='neg_log_loss',  # Note: returns NEGATIVE log loss
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

# Convert back to positive log loss (sklearn returns negative for maximization)
train_log_loss = -train_scores  # Now positive (lower is better)
val_log_loss = -val_scores

# Calculate means
train_mean = np.mean(train_log_loss, axis=1)
train_std = np.std(train_log_loss, axis=1)
val_mean = np.mean(val_log_loss, axis=1)
val_std = np.std(val_log_loss, axis=1)

# Plot (lower log loss is better)
plt.figure(figsize=(8, 6))

plt.plot(train_sizes, train_mean, 'o-', label='Training', color='blue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                 alpha=0.1, color='blue')

plt.plot(train_sizes, val_mean, 'o-', label='Cross-validation', color='red')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                 alpha=0.1, color='red')

plt.xlabel('Number of training examples')
plt.ylabel('Log Loss (lower is better)')
plt.title(f'Learning Curve - Log Loss')
plt.legend(loc='upper right')  # Upper right because lower is better
plt.grid(True, alpha=0.3)

# Diagnostic with Log Loss
final_train_loss = train_mean[-1]
final_val_loss = val_mean[-1]
gap = final_train_loss - final_val_loss  # Train MINUS Val

print(f"Final Train Log Loss: {final_train_loss:.3f}")
print(f"Final Validation Log Loss: {final_val_loss:.3f}")
print(f"Gap (Train - Val): {gap:.3f}")

### Elastic Net

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.linear_model import ElasticNet
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, log_loss, average_precision_score
from scipy.stats import uniform, loguniform
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import calibration_curve
  
# HYPERPARAMETER TUNING WITH RANDOMIZED SEARCH CV
print("Tuning hyperparameters with 10-fold CV...")

# Define parameter distribution for ElasticNet
param_dict = {
    'alpha': loguniform(0.001, 0.005),  # Regularization strength
    'l1_ratio': uniform(0, 1)          # Mix between L1 and L2 (0=Ridge, 1=Lasso)
}

# Create base model
# Note: For classification with ElasticNet, we need to use:
    # from sklearn.linear_model import SGDClassifier with loss='log_loss' and penalty='elasticnet'
    # here the loss function is the negative log likelihood which in LR is mathematically equivalent to the binary cross-entropy loss
    # OR use LogisticRegression with penalty='elasticnet' (requires solver='saga')
    # note: the optimizer is NOT ILRS (like for normal LR), but rather SGD. we can't use IRLS because the L1 penalty is not differentiable using that.

base_model = SGDClassifier(
    loss='log_loss',  # Logistic regression
    penalty='elasticnet',
    max_iter=10000,
    random_state=seed
)

# Create RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dict,
    n_iter=100,  # Number of parameter combinations to try
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
    # StratifiedKFold accounts for possible class imbalance, which regular KFold() doesn't do
    scoring='neg_log_loss',
    # optimize using binary cross entropy loss directly instead of AUC so we don't strictly need to calibrate after.
    random_state=seed,
    n_jobs=-1,  # Use all available cores
    verbose=0
)

# Fit random search
random_search.fit(X_train_scaled, y_train)
# random_search now contains:
    # All models trained during CV
    # The best parameters found
    # The best model retrained on ALL training data

# Get best model
best_model = random_search.best_estimator_ # this calls up the best model retrained on ALL training data
best_params = random_search.best_params_
best_score = random_search.best_score_

print(f"Best parameters: {best_params}")
print(f"Best CV Log Loss: {-best_score:.3f}")  # best_score is negative log loss, so negate it for positive value

# get feature importance
explainer = shap.LinearExplainer(best_model, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)
# store into dict
shap_dict['Elastic Net'] = shap_values

# Tune the decision threshold on the best model
# get probabilities for 30 different CV runs of the same (best) model
y_proba_thresh = cross_val_predict(
    best_model, X_train_scaled, y_train, 
    method='predict_proba', cv=StratifiedKFold(30, shuffle=True, random_state=seed)
)[:, 1] 

# Threshold tuning
# use the 30 probabilities to compute a bunch of F1 scores
# identify the best f1 score and designate it as the best cutoff
precision, recall, thresh = precision_recall_curve(y_train, y_proba_thresh)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9) 
optimal_threshold = thresh[np.argmax(f1_scores)] # argmax() gives the index position, not the scalar f1 value, of the highest f1 score

# store into list
for t, f1 in zip(thresh, f1_scores):
    threshold_list_row = {
        'algorithm': 'Elastic Net',
        'threshold': t,
        'f1_scores': f1
    }
    threshold_list.append(threshold_list_row)

# EVALUATE ON TEST SET
# Get predictions
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]  # Probabilities for class 1 (dead at 3m)
y_pred = (y_pred_proba >= optimal_threshold).astype(int)  # Class predictions (0/1)

# Calculate metrics
test_log_loss = log_loss(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)
# Get ROC curve data
roc_auc = roc_auc_score(y_test, y_pred_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
# Get precision-recall curve data
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba) 
# Get data for calibration analyses
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=5, strategy='uniform')
final_citl, final_slope = bootstrap_calibration_metrics(y_test, y_pred_proba)

print(f"Test Log Loss: {test_log_loss:.3f}")
print(f"Test F1: {f1:.3f}")
print(f"Optimal Threshold: {optimal_threshold:.3f}")
print(f"Test ROC-AUC: {roc_auc:.3f}")
print(f"Test PR-AUC: {pr_auc:.3f}")
print(f"Test Calibration CITL: {final_citl}")
print(f"Test Calibration Slope: {final_slope}")

# Store results
results_table.append({
    'algorithm': 'Elastic Net',
    'log_loss': test_log_loss,
    'f1': f1,
    'optimal_threshold': optimal_threshold,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'final_citl': final_citl,
    'final_slope': final_slope
})

# Append to storage list for plotting graph
results_graph.append({
    'algorithm': 'Elastic Net',
    'roc_auc': roc_auc,
    'fpr': fpr,
    'tpr': tpr,
    'pr_auc': pr_auc,
    'precision': precision,
    'recall': recall,
    'prob_true': prob_true,
    'prob_pred': prob_pred
})

# Append to pred and proba dicts for running delong and mcnemar tests
models_pred['Elastic Net'] = y_pred
models_proba['Elastic Net'] = y_pred_proba

In [ ]:
plt.hist(y_pred_proba, bins=15)
plt.title("enet predicted probabilities")

In [ ]:
# ===========================
# Learning Curve for Log Loss
# ===========================

from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np

# Generate learning curve with log loss
train_sizes, train_scores, val_scores = learning_curve(
    best_model, 
    X_train_scaled, 
    y_train,
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
    scoring='neg_log_loss',  # Note: returns NEGATIVE log loss
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

# Convert back to positive log loss (sklearn returns negative for maximization)
train_log_loss = -train_scores  # Now positive (lower is better)
val_log_loss = -val_scores

# Calculate means
train_mean = np.mean(train_log_loss, axis=1)
train_std = np.std(train_log_loss, axis=1)
val_mean = np.mean(val_log_loss, axis=1)
val_std = np.std(val_log_loss, axis=1)

# Plot (lower log loss is better)
plt.figure(figsize=(8, 6))

plt.plot(train_sizes, train_mean, 'o-', label='Training', color='blue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                 alpha=0.1, color='blue')

plt.plot(train_sizes, val_mean, 'o-', label='Cross-validation', color='red')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                 alpha=0.1, color='red')

plt.xlabel('Number of training examples')
plt.ylabel('Log Loss (lower is better)')
plt.title(f'Learning Curve - Log Loss')
plt.legend(loc='upper right')  # Upper right because lower is better
plt.grid(True, alpha=0.3)

# Diagnostic with Log Loss
final_train_loss = train_mean[-1]
final_val_loss = val_mean[-1]
gap = final_train_loss - final_val_loss  # Train MINUS Val

print(f"Final Train Log Loss: {final_train_loss:.3f}")
print(f"Final Validation Log Loss: {final_val_loss:.3f}")
print(f"Gap (Train - Val): {gap:.3f}")

### Support Vector Machine

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, precision_recall_curve, average_precision_score, log_loss 
from scipy.stats import uniform, loguniform
from sklearn.calibration import calibration_curve

# HYPERPARAMETER TUNING WITH RANDOMIZED SEARCH CV
print("Tuning hyperparameters with 10-fold CV...")

# Define parameter distribution
param_dict = {
    'C': loguniform(0.02, 10.0),    # regularization parameter, default=1. common to all SVM kernels, trades off misclassification of training examples against simplicity of the decision surface. A low C makes the decision surface smooth, while a high C aims at classifying all training examples correctly.
    'gamma': loguniform(0.9, 1.0),   # kernel coefficient, defines how much influence a single training example has. The larger gamma is, the closer other examples must be to be affected.
    'degree': range(2, 6, 1),    # only relevant for polynomial kernel
    'coef0': uniform(0, 1),   # only relevant for polynomial and sigmoid kernels
    'kernel': ['linear']
}
    
# IMPORTANT: SVC doesn't predict probabilities by default
# Create base model with probability=True  
base_model = SVC(
    probability=True,  # This is crucial! Without it, SVC doesn't have the predict_proba method. It uses Platt scaling to calibrate probabilities, which takes extra computation time but is necessary for ROC/PR curves.
    max_iter=10000,
    random_state=seed,
    class_weight={0:1, 1:15}  # OPTIONAL: handle class imbalance
)
    
# Create RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dict,
    n_iter=100,  # Number of parameter combinations to try
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
    # StratifiedKFold accounts for possible class imbalance, which regular KFold() doesn't do
    scoring='neg_log_loss',
    # optimize using binary cross entropy loss directly instead of AUC so we don't strictly need to calibrate after.
    random_state=seed,
    n_jobs=-1,  # Use all available cores
    verbose=0
)
    
# Fit random search
random_search.fit(X_train_scaled, y_train)
# random_search now contains:
    # All models trained during CV
    # The best parameters found
    # The best model retrained on ALL training data
    
# Get best model
best_model = random_search.best_estimator_ # this calls up the best model retrained on ALL training data
best_params = random_search.best_params_
best_score = random_search.best_score_

print(f"Best parameters: {best_params}")
print(f"Best CV Log Loss: {-best_score:.3f}")  # best_score is negative log loss, so negate it for positive value

# get feature importance
# Non-linear, non-tree → KernelExplainer
# Why Kernel/Poly SVM Needs KernelExplainer instead of LinearExplainer: SVM with RBF kernel maps data to a higher-dimensional space. There are no simple coef_values to extract like with linear models (LR/enet). 
# KernelExplainer works by:
    # Sampling combinations of features (masking some, keeping others)
    # Evaluating model predictions with each masked version
    # Using game theory to distribute credit
    # This is slower but works for any black-box model
# What is "background"? 
    # KernelExplainer needs a reference dataset to compute the "baseline" prediction
    # Normally, you'd pass the entire training set (X_train_scaled). If training set has 2,000 patients, every SHAP calculation references all 2,000 patients. But this is very expensive.
    # What shap.kmeans(X_train_scaled, 50) does:
        # Runs K-Means clustering on your training data
        # Finds 50 representative "average" patients (cluster centers)
        # Returns a small matrix of shape (50, n_features)
        # Why this works: The baseline prediction is just the average model output over the background. Using 50 well-chosen representative samples gives almost the same average as using all 2,000 samples, but is 40× faster .
# Random sampling instead of shap.kmeans as the code is just broken due to package updates and incompatibility etc
rng = np.random.default_rng(seed) # Creates a random number generator with a fixed seed
background_idx = rng.choice(len(X_train_scaled), size=50, replace=False)
# len(X_train_scaled): Pick from integers 0 to 1386 (all row positions)
# size=50: Pick exactly 50 numbers
# replace=False: Don't pick the same row twice (no duplicates)
background = X_train_scaled.iloc[background_idx] # if you only provide 1 argument to iloc, it still works but it just assumes you want row slicing
# We need to give Kernel Explainer a prediction function (model.predict_proba) instead of just the model object, like for Linear Explainer and Tree Explainer. Because Kernel Explainer treats the model as a black box. It cannot look inside to extract coefs and stuff. It can only ask: "What would you predict for this input?" so it needs the prediction function, which it uses to keep trying and testing predictions on those 50 samples.
# When computing SHAP for a patient, it:
    # Creates hundreds of "masked" versions of that patient (some features hidden, some kept)
    # Calls model.predict_proba(masked_patient) for each masked version
    # Uses game theory (Shapley values) to distribute credit based on how predictions change
explainer = shap.KernelExplainer(best_model.predict_proba, background)
# Subset and use first 100 test samples. Instead of computing SHAP for all 500 test patients, only compute for the first 100. Reduces computation by another 5×. Why this is acceptable:
    # SHAP importance (mean absolute SHAP) converges after ~50-100 patients
    # The feature ranking stabilizes quickly; you don't need the full test set for importance analysis
shap_values = explainer.shap_values(X_test_scaled.iloc[:100])
# KernelExplainer returns a list [class0_array, class1_array] for binary classification
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Take class 1 (death)
elif shap_values.ndim == 3: # if 3D array (x, x, 2)
    shap_values = shap_values[:, :, 1] # Take class 1 (death)
# store into dict
shap_dict['Support Vector Machine'] = shap_values

# Tune the decision threshold on the best model
# get probabilities for 30 different CV runs of the same (best) model
y_proba_thresh = cross_val_predict(
    best_model, X_train_scaled, y_train, 
    method='predict_proba', cv=StratifiedKFold(30, shuffle=True, random_state=seed)
)[:, 1] 

# Threshold tuning
# use the 30 probabilities to compute a bunch of F1 scores
# identify the best f1 score and designate it as the best cutoff
precision, recall, thresh = precision_recall_curve(y_train, y_proba_thresh)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9) 
optimal_threshold = thresh[np.argmax(f1_scores)] # argmax() gives the index position, not the scalar f1 value, of the highest f1 score

# store into list
for t, f1 in zip(thresh, f1_scores):
    threshold_list_row = {
        'algorithm': 'Support Vector Machine',
        'threshold': t,
        'f1_scores': f1
    }
    threshold_list.append(threshold_list_row)

# EVALUATE ON TEST SET
# Get predictions
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]  # Probabilities for class 1 (dead at 3m), now works with probability=True
y_pred = (y_pred_proba >= optimal_threshold).astype(int)  # Class predictions (0/1)

# Calculate metrics
test_log_loss = log_loss(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)
# Get ROC curve data
roc_auc = roc_auc_score(y_test, y_pred_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
# Get precision-recall curve data
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba) 
# Get data for calibration analyses
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=5, strategy='uniform')
final_citl, final_slope = bootstrap_calibration_metrics(y_test, y_pred_proba)

print(f"Test Log Loss: {test_log_loss:.3f}")
print(f"Test F1: {f1:.3f}")
print(f"Optimal Threshold: {optimal_threshold:.3f}")
print(f"Test ROC-AUC: {roc_auc:.3f}")
print(f"Test PR-AUC: {pr_auc:.3f}")
print(f"Test Calibration CITL: {final_citl}")
print(f"Test Calibration Slope: {final_slope}")

# Store results
results_table.append({
    'algorithm': 'Support Vector Machine',
    'log_loss': test_log_loss,
    'f1': f1,
    'optimal_threshold': optimal_threshold,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'final_citl': final_citl,
    'final_slope': final_slope
})

# Append to storage list for plotting graph
results_graph.append({
    'algorithm': 'Support Vector Machine',
    'roc_auc': roc_auc,
    'fpr': fpr,
    'tpr': tpr,
    'pr_auc': pr_auc,
    'precision': precision,
    'recall': recall,
    'prob_true': prob_true,
    'prob_pred': prob_pred
})

# Append to pred and proba dicts for running delong and mcnemar tests
models_pred['Support Vector Machine'] = y_pred
models_proba['Support Vector Machine'] = y_pred_proba

In [ ]:
plt.hist(y_pred_proba, bins=15)
plt.title("LR predicted probabilities")

In [ ]:
# ===========================
# Learning Curve for Log Loss
# ===========================

from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np

# Generate learning curve with log loss
train_sizes, train_scores, val_scores = learning_curve(
    best_model, 
    X_train_scaled, 
    y_train,
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
    scoring='neg_log_loss',  # Note: returns NEGATIVE log loss
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

# Convert back to positive log loss (sklearn returns negative for maximization)
train_log_loss = -train_scores  # Now positive (lower is better)
val_log_loss = -val_scores

# Calculate means
train_mean = np.mean(train_log_loss, axis=1)
train_std = np.std(train_log_loss, axis=1)
val_mean = np.mean(val_log_loss, axis=1)
val_std = np.std(val_log_loss, axis=1)

# Plot (lower log loss is better)
plt.figure(figsize=(8, 6))

plt.plot(train_sizes, train_mean, 'o-', label='Training', color='blue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                 alpha=0.1, color='blue')

plt.plot(train_sizes, val_mean, 'o-', label='Cross-validation', color='red')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                 alpha=0.1, color='red')

plt.xlabel('Number of training examples')
plt.ylabel('Log Loss (lower is better)')
plt.title(f'Learning Curve - Log Loss')
plt.legend(loc='upper right')  # Upper right because lower is better
plt.grid(True, alpha=0.3)

# Diagnostic with Log Loss
final_train_loss = train_mean[-1]
final_val_loss = val_mean[-1]
gap = final_train_loss - final_val_loss  # Train MINUS Val

print(f"Final Train Log Loss: {final_train_loss:.3f}")
print(f"Final Validation Log Loss: {final_val_loss:.3f}")
print(f"Gap (Train - Val): {gap:.3f}")

### XGBoost

In [ ]:

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, auc, log_loss, make_scorer
from scipy.stats import uniform, loguniform

# HYPERPARAMETER TUNING WITH RANDOMIZED SEARCH CV
print("Tuning hyperparameters with 10-fold CV...")

# Define parameter distribution for XGBoost
param_dict = {
    # Number of boosting rounds (trees)
    'n_estimators': range(700, 1300, 50),  # or use: np.arange(100, 501, 100)
    
    # Learning rate (eta) shrinks the contribution of each tree as we get nearer to the minimum:
        # New prediction = Previous prediction + learning_rate × (new tree prediction)
    'learning_rate': uniform(0.005, 0.030),  # typical range: 0.01-0.3
    
    # Maximum depth of each tree - controls model complexity
    'max_depth': range(9,16),  # typical range: 3-10
    
    # Minimum child weight - don't create a node with fewer than min_child_weight samples
    'min_child_weight': range(5, 10),  # larger = more conservative
    
    # Subsample ratio of training instances (row sampling)
    'subsample': uniform(loc=0.2, scale=0.5),  # lower bound = 0.6, varies over 0.4, upper bound = 1.0
    
    # Subsample ratio of columns (feature sampling) for each tree
    'colsample_bytree': uniform(loc=0.1, scale=0.9),  # range: 0.6-1.0
    
    # L1 regularization term on weights (alpha)
    'reg_alpha': loguniform(1e-5, 0.1),  # range: very small to 10
    
    # L2 regularization term on weights (lambda). This acts as a smoothing filter that reduces noise but keeps all features, used when we want to prevent any single feature from dominating. L1 and L2 can be used together as they complement each other. L1 removes completely irrelevant features, while L2 controls the impact of relevant features.
    'reg_lambda': loguniform(1e-5, 0.1),  # range: very small to 10
    
    # Gamma - minimum loss reduction required to make a further partition
    'gamma': loguniform(1e-5, 0.1)  # range: very small to 1
}
    
# Create base model  
base_model = xgb.XGBClassifier(
    objective='binary:logistic',  # for binary classification. Uses logistic loss function (binary cross-entropy, Applies sigmoid transformation to raw predictions, Outputs probabilities that sum to 1
    eval_metric='logloss',  # evaluation metric, not actually used here because the CV uses scoring='roc_auc' for optimization. The eval_metric would only be used if you implemented early stopping.
    base_score=0.5, # When XGBoost builds trees, it starts from a global bias: probability of 0.5 for everyone. Then each tree adds to this initial prediction. base_score=0.5 means: "Without any information, I assume a 50% chance of death." This is the default and almost always correct for binary classification. So we explicitly state 0.5 here, otherwise when RandomizedSearchCV clones and fits the model, the base_score gets serialized as '[5E-1]' (a string representation of 0.5 in scientific notation). Older SHAP versions can't parse this.
    scale_pos_weight=1.15, # Scale positive weights - for imbalanced datasets. DO NOT randomly search this! Calculate it exactly, this should be a fixed value based on your data. The optimal scale_pos_weight has a theoretical basis: typically scale_pos_weight = (negative samples) / (positive samples). This balances the loss function so that the model pays equal attention to both classes. If you search randomly, and scale_pos_weight is too low, the model still ignores positives. if it's too high, the model becomes obssessed with positives, resulting in tons of false positives
    random_state=seed,
    n_jobs=1  # XGBoost has its own parallelization, so set this to 1 when using n_jobs=-1 in CV
)
    
# Create RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dict,
    n_iter=100,  # XGBoost has more parameters, so increase iterations
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
    # StratifiedKFold accounts for possible class imbalance, which regular KFold() doesn't do
    scoring={
        'neg_log_loss': 'neg_log_loss', # shorthand. does a lookup to get the full make_scorer call under the hood.
        'roc_auc': make_scorer(roc_auc_score, needs_proba=True),
        'pr_auc': make_scorer(average_precision_score, needs_proba=True)
    },
    refit='neg_log_loss',  # only uses one metric to select the best model, but all metrics are computed and stored in cv_results_. we optimize using binary cross entropy loss directly instead of AUC so we don't strictly need to calibrate after.
    random_state=seed,
    n_jobs=-1,  # Use all available cores
    verbose=0
)
    
# Fit random search
random_search.fit(X_train_scaled, y_train)
# random_search now contains:
    # All models trained during CV
    # The best parameters found
    # The best model retrained on ALL training data
    
# GET MEAN SCORES (averaged across CV folds)
print(f"Best Hyperparameters: {random_search.best_params_}")
print(f"Mean ROC_AUC: {random_search.cv_results_['mean_test_roc_auc'][random_search.best_index_]}")
print(f"Mean PR_AUC: {random_search.cv_results_['mean_test_pr_auc'][random_search.best_index_]}")
print(f"Mean Log Loss: {-random_search.cv_results_['mean_test_neg_log_loss'][random_search.best_index_]}")
# random_search.cv_results_['mean_test_roc_auc'] → array of length 100 (means for each hyperparameter combo). there's 100 because n_iter=100 in RandomizedSearchCV()
# random_search.best_index_  # Returns the integer index of the best hyperparameter combination
# random_search.cv_results_['mean_test_roc_auc'][random_search.best_index_] → the mean ROC AUC for the best combo

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
import xgboost as xgb
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, precision_recall_curve, log_loss, average_precision_score
from scipy.stats import uniform, loguniform
from sklearn.calibration import calibration_curve

   
# Get best model
best_model = random_search.best_estimator_ # this calls up the best model retrained on ALL training data
best_params = random_search.best_params_
best_score = random_search.best_score_

print(f"Best parameters: {best_params}")
print(f"Best CV Log Loss: {-best_score:.3f}")  # best_score is negative log loss, so negate it for positive value

# get feature importance
rng = np.random.default_rng(seed)
background_idx = rng.choice(len(X_train_scaled), size=50, replace=False)
background = X_train_scaled.iloc[background_idx]

explainer = shap.Explainer(best_model.predict_proba, background)
shap_result = explainer(X_test_scaled.iloc[:500])

# Extract the SHAP values array
if hasattr(shap_result, 'values'):
    shap_values = shap_result.values
    # For binary classification, shape of the output produced by the explainer() is in this array form (n_samples, n_features, 2)
    # e.g. (500, 61, 2): 500 samples because you specified so, 61 features -> these make up a standard 2d array. but because we have 2 outcomes, each outcome has one 2d array. so here we have two 2d arrays stacked together, making a 3d array.
    if len(shap_values.shape) == 3:
        shap_values = shap_values[:, :, 1]  # we slice to get only the 2d array for Class 1
    shap_dict['XGBoost'] = shap_values

# Tune the decision threshold on the best model
# get probabilities for 30 different CV runs of the same (best) model
y_proba_thresh = cross_val_predict(
    best_model, X_train_scaled, y_train, 
    method='predict_proba', cv=StratifiedKFold(30, shuffle=True, random_state=seed)
)[:, 1] 

# Threshold tuning
# use the 30 probabilities to compute a bunch of F1 scores
# identify the best f1 score and designate it as the best cutoff
precision, recall, thresh = precision_recall_curve(y_train, y_proba_thresh)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9) 
optimal_threshold = thresh[np.argmax(f1_scores)] # argmax() gives the index position, not the scalar f1 value, of the highest f1 score

# store into list
for t, f1 in zip(thresh, f1_scores):
    threshold_list_row = {
        'algorithm': 'XGBoost',
        'threshold': t,
        'f1_scores': f1
    }
    threshold_list.append(threshold_list_row)

# EVALUATE ON TEST SET

# ===============
# Get predictions
# ===============
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]  # Probabilities for class 1 (dead at 3m), now works with probability=True
y_pred = (y_pred_proba >= optimal_threshold).astype(int)  # Class predictions (0/1)

# =================
# Calculate metrics
# =================
test_log_loss = log_loss(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)

# Get ROC curve data
roc_auc = roc_auc_score(y_test, y_pred_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# Get precision-recall curve data
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_pred_proba)
# The PR curve is step-shaped. The curve produced by precision_recall_curve() is not linear between points.
pr_auc = average_precision_score(y_test, y_pred_proba) 
# do not use auc() here because it assumes linear interpolation; it connects points with straight lines (trapezoids).
# whereas average_precision_score() uses step-wise interpolation. It matches how PR curves are actually visualized (step-shaped curves) and defined.

# =================================
# Get data for calibration analyses
# =================================
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=5, strategy='uniform')
final_citl, final_slope = bootstrap_calibration_metrics(y_test, y_pred_proba)

print(f"Test Log Loss: {test_log_loss:.3f}")
print(f"Test F1: {f1:.3f}")
print(f"Optimal Threshold: {optimal_threshold:.3f}")
print(f"Test ROC-AUC: {roc_auc:.3f}")
print(f"Test PR-AUC: {pr_auc:.3f}")
print(f"Test Calibration CITL: {final_citl}")
print(f"Test Calibration Slope: {final_slope}")

# Store results
results_table.append({
    'algorithm': 'XGBoost',
    'log_loss': test_log_loss,
    'f1': f1,
    'optimal_threshold': optimal_threshold,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'final_citl': final_citl,
    'final_slope': final_slope
})

# Append to storage list for plotting graph
results_graph.append({
    'algorithm': 'XGBoost',
    'roc_auc': roc_auc,
    'fpr': fpr,
    'tpr': tpr,
    'pr_auc': pr_auc,
    'precision': precision,
    'recall': recall,
    'prob_true': prob_true,
    'prob_pred': prob_pred
})

# Append to pred and proba dicts for running delong and mcnemar tests
models_pred['XGBoost'] = y_pred
models_proba['XGBoost'] = y_pred_proba

In [ ]:
plt.hist(y_pred_proba, bins=15)
plt.title("LR predicted probabilities")

In [ ]:
# ===========================
# Learning Curve for Log Loss
# ===========================

from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np

# Generate learning curve with log loss
train_sizes, train_scores, val_scores = learning_curve(
    best_model, 
    X_train_scaled, 
    y_train,
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
    scoring='neg_log_loss',  # Note: returns NEGATIVE log loss
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

# Convert back to positive log loss (sklearn returns negative for maximization)
train_log_loss = -train_scores  # Now positive (lower is better)
val_log_loss = -val_scores

# Calculate means
train_mean = np.mean(train_log_loss, axis=1)
train_std = np.std(train_log_loss, axis=1)
val_mean = np.mean(val_log_loss, axis=1)
val_std = np.std(val_log_loss, axis=1)

# Plot (lower log loss is better)
plt.figure(figsize=(8, 6))

plt.plot(train_sizes, train_mean, 'o-', label='Training', color='blue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                 alpha=0.1, color='blue')

plt.plot(train_sizes, val_mean, 'o-', label='Cross-validation', color='red')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                 alpha=0.1, color='red')

plt.xlabel('Number of training examples')
plt.ylabel('Log Loss (lower is better)')
plt.title(f'Learning Curve - Log Loss')
plt.legend(loc='upper right')  # Upper right because lower is better
plt.grid(True, alpha=0.3)

# Diagnostic with Log Loss
final_train_loss = train_mean[-1]
final_val_loss = val_mean[-1]
gap = final_train_loss - final_val_loss  # Train MINUS Val

print(f"Final Train Log Loss: {final_train_loss:.3f}")
print(f"Final Validation Log Loss: {final_val_loss:.3f}")
print(f"Gap (Train - Val): {gap:.3f}")

### Adaboost

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier  # AdaBoost's default base estimator
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, precision_recall_curve, average_precision_score, log_loss
from scipy.stats import uniform, loguniform, randint
from sklearn.calibration import calibration_curve

# HYPERPARAMETER TUNING WITH RANDOMIZED SEARCH CV
print("Tuning hyperparameters with 10-fold CV...")

# Define parameter distribution for AdaBoost
param_dict = {
    # Number of boosting stages (estimators)
    'n_estimators': randint(150, 1300),  # picks any random value from 50 to 999
    
    # Learning rate - shrinks the contribution of each classifier
    'learning_rate': loguniform(0.01, 2),  # range: 0.01 to 2.0 (typical)
    
    # Algorithm - the boosting algorithm to use
    'algorithm': ['SAMME.R'],  # SAMME.R uses probabilities, usually faster. typically recommended to go with SAMME.R unless the base estimator does not have a predict_proba() method, then we would have to use SAMME
    
    # Base estimator parameters (if using default DecisionTree)
    # Note: These need 'estimator__' prefix when tuning base estimator params
    'estimator__max_depth': range(2, 8),  # Depth of decision stumps/trees
    'estimator__min_samples_split': range(16, 24),  # Min samples needed at node for split to be attempted
}
    
# Create base model with default base estimator (DecisionTreeClassifier)
# Note: We need to create a DecisionTreeClassifier and pass it to AdaBoost
base_estimator = DecisionTreeClassifier(
    class_weight='balanced',  # Handle imbalance here in the base estimator, as AdaBoost doesn't have a direct scale_pos_weight equivalent in the base model specification. 
    random_state=seed
    # We'll tune its parameters via param_dict with 'estimator__' prefix
)

base_model = AdaBoostClassifier(
    estimator=base_estimator,  # Base estimator (default is DecisionTree)
    random_state=seed
)

# Create RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dict,
    n_iter=60,  # Number of parameter combinations to try
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
    scoring={
        'neg_log_loss': 'neg_log_loss',
        'pr_auc': make_scorer(average_precision_score, needs_proba=True),
        'roc_auc': make_scorer(roc_auc_score, needs_proba=True)
    },
    refit='neg_log_loss',  # Optimize log loss. Reference this by the name you gave in the 'scoring' argument
    random_state=seed,
    n_jobs=-1,  # Use all available cores
    verbose=0
)
    
# Fit random search
random_search.fit(X_train_scaled, y_train)
# random_search now contains:
    # All models trained during CV
    # The best parameters found
    # The best model retrained on ALL training data

# get the best params and metrics from CV
print(f"Best hyperparameters: {random_search.best_params_}")
print(f"Best Log Loss from CV: {-random_search.cv_results_['mean_test_neg_log_loss'][random_search.best_index_]}")
print(f"Best PR-AUC from CV: {random_search.cv_results_['mean_test_pr_auc'][random_search.best_index_]}")
print(f"Best ROC-AUC from CV: {random_search.cv_results_['mean_test_roc_auc'][random_search.best_index_]}")
# to access keys from cv_results_ we have to use: prefix 'mean_test_' + scorer name. it follows a set pattern

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier  # AdaBoost's default base estimator
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, precision_recall_curve, average_precision_score, log_loss
from scipy.stats import uniform, loguniform, randint
from sklearn.calibration import calibration_curve

   
# Get best model
best_model = random_search.best_estimator_ # this calls up the best model retrained on ALL training data
best_params = random_search.best_params_
best_score = random_search.best_score_

print(f"Best parameters: {best_params}")
print(f"Best CV Log Loss: {-best_score:.3f}")  # best_score is negative log loss, so negate it for positive value

# get feature importance
# Random sampling instead of shap.kmeans
rng = np.random.default_rng(seed)
background_idx = rng.choice(len(X_train_scaled), size=50, replace=False)
background = X_train_scaled.iloc[background_idx]
# cannot use Tree Explainer even though this algorithm is tree-based. The reason AdaBoost is treated differently comes down to its ensemble structure. AdaBoost combines "weak learners" (often shallow decision trees) by assigning a weight to each tree's vote based on its accuracy. The algorithms that TreeExplainer natively supports (like Random Forest and XGBoost) either use simple averaging or gradient-based weighting, which SHAP's internal code is specifically designed to parse. we will use Kernel Explainer which will work with anything.
explainer = shap.KernelExplainer(best_model.predict_proba, background)
shap_values = explainer.shap_values(X_test_scaled.iloc[:100])
# KernelExplainer returns a list [class0_array, class1_array] for binary classification
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Take class 1 (death)
elif shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1] # Take class 1 (death)
# store into dict
shap_dict['AdaBoost'] = shap_values

# Tune the decision threshold on the best model
# get probabilities for 30 different CV runs of the same (best) model
y_proba_thresh = cross_val_predict(
    best_model, X_train_scaled, y_train, 
    method='predict_proba', cv=StratifiedKFold(30, shuffle=True, random_state=seed)
)[:, 1] 

# Threshold tuning
# use the 30 probabilities to compute a bunch of F1 scores
# identify the best f1 score and designate it as the best cutoff
precision, recall, thresh = precision_recall_curve(y_train, y_proba_thresh)
    # internally, precision_recall_curve() gets the unique list of y_proba_thresh values, so you don't need to preprocess with np.unique() before feeding the array into this method. just feed in the full (may be non-unique) list of probabilities, otherwise it might throw an error if y_proba_thresh is not the same length as y_train
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9) 
    # add small offset to prevent divison by zero, will not affect results
    # The last precision/recall value corresponds to threshold=infinity when you predict NOTHING as positive. we don't need that, so we extract everything except that
optimal_threshold = thresh[np.argmax(f1_scores)] # argmax() gives the index position, not the scalar f1 value, of the highest f1 score

# store into list
for t, f1 in zip(thresh, f1_scores):
    threshold_list_row = {
        'algorithm': 'AdaBoost',
        'threshold': t,
        'f1_scores': f1
    }
    threshold_list.append(threshold_list_row)

# EVALUATE ON TEST SET
# Get predictions
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]  # Probabilities for class 1 (dead at 3m), now works with probability=True
y_pred = (y_pred_proba >= optimal_threshold).astype(int)  # Class predictions (0/1)
    # we cannot use the .predict() method because the default threshold for that is 0.5 and it cannot be changed. so we have to do this manually
    # y_pred_proba >= optimal_threshold: this gives a Boolean list of T/F. we then use .astype(int) to change it to a 0/1 list, which looks like 0/1 class predictions given by .predict(), we can then input this into f1_score(). actually f1_score() also accepts a boolean list as it can convert them on its own, but let's do this to be safe.

# Calculate metrics
test_log_loss = log_loss(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)
# Get ROC curve data
roc_auc = roc_auc_score(y_test, y_pred_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
# Get precision-recall curve data
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba) 
# Get data for calibration analyses
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=5, strategy='uniform')
    # 'prob_pred': Mean predicted probabilities (from calibration curve)
    # 'prob_true': Actual fraction of positives (from calibration curve)
    # n_bins should be 5-10 for small data. only go up to 20-30 if you have big n, otherwise the curve gets very jagged and noisy because too few samples in each bin.
    # strategy='quantile': Splits data so each bin has roughly equal no of samples
final_citl, final_slope = bootstrap_calibration_metrics(y_test, y_pred_proba)

print(f"Test Log Loss: {test_log_loss:.3f}")
print(f"Test F1: {f1:.3f}")
print(f"Optimal Threshold: {optimal_threshold:.3f}")
print(f"Test ROC-AUC: {roc_auc:.3f}")
print(f"Test PR-AUC: {pr_auc:.3f}")
print(f"Test Calibration CITL: {final_citl}")
print(f"Test Calibration Slope: {final_slope}")

# Store results
results_table.append({
    'algorithm': 'AdaBoost',
    'log_loss': test_log_loss,
    'f1': f1,
    'optimal_threshold': optimal_threshold,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'final_citl': final_citl,
    'final_slope': final_slope
})

# Append to storage list for plotting graph
results_graph.append({
    'algorithm': 'AdaBoost',
    'roc_auc': roc_auc,
    'fpr': fpr,
    'tpr': tpr,
    'pr_auc': pr_auc,
    'precision': precision,
    'recall': recall,
    'prob_true': prob_true,
    'prob_pred': prob_pred
})

# Append to pred and proba dicts for running delong and mcnemar tests
models_pred['AdaBoost'] = y_pred
models_proba['AdaBoost'] = y_pred_proba

In [ ]:
plt.hist(y_pred_proba, bins=15)
plt.title("LR predicted probabilities")

In [ ]:
# ===========================
# Learning Curve for Log Loss
# ===========================

from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np

# Generate learning curve with log loss
train_sizes, train_scores, val_scores = learning_curve(
    best_model, 
    X_train_scaled, 
    y_train,
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
    scoring='neg_log_loss',  # Note: returns NEGATIVE log loss
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

# Convert back to positive log loss (sklearn returns negative for maximization)
train_log_loss = -train_scores  # Now positive (lower is better)
val_log_loss = -val_scores

# Calculate means
train_mean = np.mean(train_log_loss, axis=1)
train_std = np.std(train_log_loss, axis=1)
val_mean = np.mean(val_log_loss, axis=1)
val_std = np.std(val_log_loss, axis=1)

# Plot (lower log loss is better)
plt.figure(figsize=(8, 6))

plt.plot(train_sizes, train_mean, 'o-', label='Training', color='blue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                 alpha=0.1, color='blue')

plt.plot(train_sizes, val_mean, 'o-', label='Cross-validation', color='red')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                 alpha=0.1, color='red')

plt.xlabel('Number of training examples')
plt.ylabel('Log Loss (lower is better)')
plt.title(f'Learning Curve - Log Loss')
plt.legend(loc='upper right')  # Upper right because lower is better
plt.grid(True, alpha=0.3)

# Diagnostic with Log Loss
final_train_loss = train_mean[-1]
final_val_loss = val_mean[-1]
gap = final_train_loss - final_val_loss  # Train MINUS Val

print(f"Final Train Log Loss: {final_train_loss:.3f}")
print(f"Final Validation Log Loss: {final_val_loss:.3f}")
print(f"Gap (Train - Val): {gap:.3f}")

### Decision Threshold and F1 Score

In [ ]:
# look at all the threshold-f1 pairs tested for the best model in each algorithm
threshold_df = pd.DataFrame(threshold_list)
threshold_df

### Plot Graphs on Test Data

In [ ]:
# stack results_graph into a df
results_graph_df = pd.DataFrame(results_graph)
results_graph_df

In [ ]:
# subset to relevant results only
results_graph_df = results_graph_df.drop_duplicates(subset='algorithm',keep='last').reset_index(drop=True)
results_graph_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

# Plot both
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# set colors
colors = plt.cm.tab10(np.linspace(0, 1, len(results_graph_df)))
# plt.cm.tab10 is a colormap with 10 distinct colors (good for categorical data)
    # Colors in colormaps are indexed by floats between 0 and 1, not integers
    # plt.cm.tab10 is a "Qualitative" colormap that INTERPOLATES between colors. It creates a BLEND between the two nearest colors

# ROC Curve
for idx, row in results_graph_df.iterrows():
    axes[0].plot(row['fpr'], row['tpr'], 
                 color=colors[idx], 
                 label=f"{row['algorithm']} (ROC-AUC = {row['roc_auc']:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--') # Plots the diagonal line from (0,0) to (1,1) in black dashed style
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3) # adds a semi-transparent grid

# Calibration Tests
for idx, row in results_graph_df.iterrows():
    axes[1].plot(row['prob_pred'], row['prob_true'], 
                 color=colors[idx], 
                 label=row['algorithm'])

axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Actual Fraction of Positives')
axes[1].set_title('Calibration Plot')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Add an overall figure title
fig.suptitle('Predictive Performance on Test Data', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### All Results

In [ ]:
# stack results_graph into a df
results_table_df = pd.DataFrame(results_table)
results_table_df

In [ ]:
# subset to relevant results only
results_table_df = results_table_df.drop_duplicates(subset='algorithm',keep='last').reset_index(drop=True)
results_table_df

### Comparison of Model Performance

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from compare_auc_delong_xu import delong_roc_test
from typing import List, Dict, Optional, Literal

# ============================================================================
# SIMPLIFIED COMPARISON FUNCTIONS
# ============================================================================

def compare_f1_mcnemar(y_test, pred1, pred2):
    """
    McNemar's test for F1 comparison

    Parameters:
    -----------
    y_test: binary values (0/1) of the outcome from the test set
    pred1: Array containing binary class predictions from Model 1 
    pred2: Array containing binary class predictions from Model 2
    """
    
    # Create 2x2 contingency table for correct/incorrect
    # Table:
    #           | Model 2 Correct | Model 2 Wrong
    # ----------|-----------------|---------------
    # Model 1   |        a        |       b
    # Correct   |                 |
    # ----------|-----------------|---------------
    # Model 1   |        c        |       d
    # Wrong     |                 |
    
    correct1 = (pred1 == y_test)
    correct2 = (pred2 == y_test)
    
    a = np.sum(correct1 & correct2)   # Both correct
    b = np.sum(correct1 & ~correct2)  # Only model 1 correct
    c = np.sum(~correct1 & correct2)  # Only model 2 correct
    d = np.sum(~correct1 & ~correct2) # Both wrong
    
    # Create contingency table
    table = [[a, b], [c, d]]
    
    # Perform McNemar test (with continuity correction)
    result = mcnemar(table, exact=False, correction=True)
    
    return result.pvalue, table


# ============================================================================
# PAIRWISE COMPARISONS WITH MULTIPLE CORRECTION
# ============================================================================

def compare_all_models(y_test, models_proba: Dict, models_pred: Dict, model_names: List,
                      correction_method: Optional[Literal['bonferroni', 'holm', 'fdr_bh']] = 'bonferroni', alpha=0.05):
    """
    Compare all models pairwise with multiple comparison correction
    
    Parameters:
    -----------
    y_test : array-like, true labels
    models_proba : dict, {name: predicted_probabilities}
    models_pred : dict, {name: binary_predictions}
    model_names : list, names of models
    correction_method : str or None
        'bonferroni', 'holm', 'fdr_bh', or None
    alpha : float, significance level
    
    Returns:
    --------
    dict: Results with corrected p-values
    """
    n_models = len(model_names)
    n_comparisons = n_models * (n_models - 1) // 2 
    # // is a floor division method. returns integer (rounds down) instead of float, like normal division does
    
    # Normalise indices so pandas doesn't misalign on label-based lookups
    to_arr = lambda x: np.asarray(x.values if hasattr(x, 'values') else x).ravel()
    y_test = to_arr(y_test)
    models_proba = {k: to_arr(v) for k, v in models_proba.items()}
    models_pred  = {k: to_arr(v) for k, v in models_pred.items()}

    print("\n" + "="*60)
    print("PAIRWISE MODEL COMPARISONS")
    print("="*60)
    print(f"Models: {', '.join(model_names)}")
    print(f"Number of comparisons: {n_comparisons}")
    print(f"Correction method: {correction_method if correction_method else 'None'}")
    print(f"Alpha: {alpha}")
    
    # Store point estimates
    auc_scores = {name: roc_auc_score(y_test, models_proba[name]) for name in model_names} # dictionary comprehension!
    f1_scores = {name: f1_score(y_test, models_pred[name]) for name in model_names}
    
    # Store raw p-values
    auc_p_values = []
    f1_p_values = []
    auc_comparisons = []
    f1_comparisons = []
    
    # Perform all pairwise comparisons
    for i in range(n_models):
        for j in range(i + 1, n_models):
            name1 = model_names[i]
            name2 = model_names[j]
            
            # AUC comparison
            p_auc = delong_roc_test(y_test, models_proba[name1], models_proba[name2])
            auc_p_values.append(p_auc)
            auc_comparisons.append({
                'Model 1': name1,
                'Model 2': name2,
                'AUC 1': auc_scores[name1],
                'AUC 2': auc_scores[name2],
                'Difference': auc_scores[name1] - auc_scores[name2],
                'p-value (raw)': p_auc
            })
            
            # F1 comparison
            p_f1, table = compare_f1_mcnemar(y_test, models_pred[name1], models_pred[name2])
            f1_p_values.append(p_f1)
            f1_comparisons.append({
                'Model 1': name1,
                'Model 2': name2,
                'F1 1': f1_scores[name1],
                'F1 2': f1_scores[name2],
                'Difference': f1_scores[name1] - f1_scores[name2],
                'Both Correct': table[0][0],
                'Only Model 1 Correct': table[0][1],
                'Only Model 2 Correct': table[1][0],
                'Both Wrong': table[1][1],
                'p-value (raw)': p_f1
            })
    
    # Apply multiple comparison correction
    if correction_method:
        # AUC correction
        _, auc_p_corrected, _, _ = multipletests(auc_p_values, alpha=alpha, method=correction_method)
        for i, p_corr in enumerate(auc_p_corrected):
            auc_comparisons[i]['p-value (corrected)'] = p_corr
            auc_comparisons[i]['Significant'] = p_corr < alpha
        
        # F1 correction
        _, f1_p_corrected, _, _ = multipletests(f1_p_values, alpha=alpha, method=correction_method)
        for i, p_corr in enumerate(f1_p_corrected):
            f1_comparisons[i]['p-value (corrected)'] = p_corr
            f1_comparisons[i]['Significant'] = p_corr < alpha
    
    return {
        'auc_scores': auc_scores,
        'f1_scores': f1_scores,
        'auc_comparisons': pd.DataFrame(auc_comparisons),
        'f1_comparisons': pd.DataFrame(f1_comparisons),
        'n_comparisons': n_comparisons,
        'correction_method': correction_method
    }

In [ ]:
model_names = ['Logistic Regression', 'Elastic Net','Support Vector Machine', 'XGBoost']

# Run comparisons with Bonferroni correction
results = compare_all_models(
    y_test,
    models_proba,
    models_pred,
    model_names,
    correction_method='bonferroni',  # Options: 'bonferroni', 'holm', 'fdr_bh', None
    alpha=0.05
)

# Access dataframes
auc_comparisons = results['auc_comparisons']
f1_comparisons = results['f1_comparisons']

# inspect
print(f'AUC:\n {auc_comparisons}')
print(f'F1:\n {f1_comparisons}')

In [ ]:
from datetime import date

# ============================================
# FEATURE IMPORTANCE RANKING
# ============================================
# Absolute SHAP ranking + Forest plot consensus

# ============================================
# STEP 2: Calculate mean absolute SHAP per feature per model
# ============================================

print("\n" + "=" * 60)
print("CALCULATING MEAN ABSOLUTE SHAP VALUES")
print("=" * 60)

# Create dataframe with features as index
shap_importance_all = pd.DataFrame(index=X_test_scaled.columns)

# Fill with mean absolute SHAP from each model
for name, shap_vals in shap_dict.items():
    shap_importance_all[name] = np.abs(shap_vals).mean(axis=0)

print(f"\nImportance matrix shape: {shap_importance_all.shape}")
print(f"(Features: {shap_importance_all.shape[0]}, Models: {shap_importance_all.shape[1]})")

# ============================================
# STEP 3: Calculate consensus ranking
# ============================================

# Calculate mean across all models
model_columns = ['Logistic Regression', 'Elastic Net', 'XGBoost', 'Support Vector Machine']
shap_importance_all['Mean_Across_Models'] = shap_importance_all[model_columns].mean(axis=1)
shap_importance_all['Std_Across_Models'] = shap_importance_all[model_columns].std(axis=1)
shap_importance_all['Min_Across_Models'] = shap_importance_all[model_columns].min(axis=1)
shap_importance_all['Max_Across_Models'] = shap_importance_all[model_columns].max(axis=1)

# Sort by mean importance (descending)
shap_importance_all = shap_importance_all.sort_values('Mean_Across_Models', ascending=False)

# Add rank column
shap_importance_all.insert(0, 'Rank', range(1, len(shap_importance_all) + 1))

# ============================================
# STEP 4: Display full results
# ============================================

print("\n" + "=" * 60)
print("FEATURE IMPORTANCE - RANKED BY MEAN |SHAP| ACROSS ALL MODELS")
print("=" * 60)

# Display columns in nice order
display_cols = ['Rank'] + model_columns + ['Mean_Across_Models', 'Std_Across_Models']
print(shap_importance_all[display_cols].round(4))

# ============================================
# STEP 5: Extract and save top 20
# ============================================

top_20 = shap_importance_all.head(20).copy()
top_20 = top_20.reset_index().rename(columns={'index': 'Feature'})

print("\n" + "=" * 60)
print("TOP 20 FEATURES - CONSENSUS RANKING")
print("=" * 60)
print(top_20[['Rank', 'Feature'] + model_columns + ['Mean_Across_Models']].round(4))

# Save to CSV
top_20.to_csv('shap_consensus_top20.csv', index=False)
print("\n Saved: shap_consensus_top20.csv")

# ============================================
# STEP 6: Forest plot for top 20
# ============================================

fig, ax = plt.subplots(figsize=(14, 12))

# Prepare data (reverse for top-down display)
plot_data = shap_importance_all.head(20).copy()
plot_data = plot_data.iloc[::-1]

# Y positions
y_pos = np.arange(len(plot_data))

# Plot individual model points
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
#          Blue        Orange           Green       Red           Purple
for i, (model, color) in enumerate(zip(model_columns, colors[:len(model_columns)])):
    x_offset = (i - len(model_columns)/2 + 0.5) * 0.12
    ax.scatter(
        plot_data[model],
        y_pos + x_offset,
        label=model,
        s=100,
        alpha=0.7,
        color=color,
        edgecolors='white',
        linewidth=1,
        zorder=5
    )

# Plot mean as diamond
ax.scatter(
    plot_data['Mean_Across_Models'],
    y_pos,
    label='Mean Across Models',
    s=200,
    marker='D',
    color='black',
    edgecolors='white',
    linewidth=1,
    zorder=10
)

# Add range lines (min to max)
for i, (idx, row) in enumerate(plot_data.iterrows()):
    ax.hlines(
        y=i, 
        xmin=row['Min_Across_Models'], 
        xmax=row['Max_Across_Models'], 
        color='gray', 
        alpha=0.4, 
        linewidth=1.5,
        zorder=1
    )

# Customize plot
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_data.index, fontsize=11)
ax.set_xlabel('Mean |SHAP| Value', fontsize=13, fontweight='bold')
ax.set_title('Top 20 Features: SHAP Importance Consensus Across Models', fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add subtle background shading for alternating rows
for i in range(0, len(plot_data), 2):
    ax.axhspan(i - 0.5, i + 0.5, alpha=0.03, color='black', zorder=0)

plt.tight_layout()
plt.savefig('shap_consensus_top20_forest.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("Saved: shap_consensus_top20_forest.png")

# ============================================
# STEP 7: Summary statistics for paper
# ============================================

print("\n" + "=" * 60)
print("SUMMARY STATISTICS FOR PAPER")
print("=" * 60)

print(f"\nTotal features analyzed: {len(shap_importance_all)}")
print(f"Models included: {', '.join(model_columns)}")
print(f"\nTop 3 features by consensus:")
for i in range(3):
    feat = shap_importance_all.iloc[i]
    print(f"  {i+1}. {feat.name}: Mean |SHAP| = {feat['Mean_Across_Models']:.4f} "
          f"(Range: {feat['Min_Across_Models']:.4f} - {feat['Max_Across_Models']:.4f})")

# ============================================
# OPTIONAL: Save full importance matrix
# ============================================

shap_importance_all.to_csv(f'shap_importance_all_features_v6_1y_mortality_{date.today()}.csv')
print("\n Saved: shap_importance_all_features.csv (complete ranking)")

### SUBGROUP ANALYSES ON THOSE WITH EFI SCORES ONLY

In [ ]:
# drop NA's in efi_score
slsr13_small = slsr10.dropna(subset=['efi_score'])
slsr13_small.info()

In [ ]:
slsr13_small.describe(include='all')

In [ ]:
# set aside some data for temporal validation

test_df_small = slsr13_small[slsr13_small['strk_y']>2021]
print(len(test_df_small))

train_df_small = slsr13_small[slsr13_small['strk_y']<=2021]
print(len(train_df_small))

In [ ]:
# initiate empty dict to store models pred and proba for delong and mcnemar tests
models_pred_small = {}
models_proba_small = {}

# initiate empty list to store all results
results_table_small = []
results_graph_small = []
threshold_list_small = []

#### WITH EFI AS FEATURE

In [ ]:
from sklearn.preprocessing import StandardScaler

# Prepare data
X_train_efi = train_df_small.drop(['id','bi0.25','bi1','death1','death0.25'], axis=1)
y_train_efi = train_df_small['death1']

X_test_efi = test_df_small.drop(['id','bi0.25','bi1','death1','death0.25'], axis=1)
y_test_efi = test_df_small['death1']
    
# Standardize
scaler = StandardScaler()
X_train_efi_scaled = scaler.fit_transform(X_train_efi)
X_test_efi_scaled = scaler.transform(X_test_efi)

# inspect
print(f'Train: {X_train_efi_scaled}')
print(f'Test: {X_test_efi_scaled}')

In [ ]:
# event rate
print(f'Out of {len(y_train_efi)} training samples, we have {y_train_efi.sum()} events, or event rate of {y_train_efi.mean()}')
print(f'Out of {len(y_test_efi)} test samples, we have {y_test_efi.sum()} events, or event rate of {y_test_efi.mean()}')

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.linear_model import ElasticNet
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, log_loss, average_precision_score
from scipy.stats import uniform, loguniform
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import calibration_curve


# HYPERPARAMETER TUNING WITH RANDOMIZED SEARCH CV
print("Tuning hyperparameters with 10-fold CV...")

# Define parameter distribution for ElasticNet
param_dict = {
    'alpha': loguniform(0.08, 1.0),  # Regularization strength
    'l1_ratio': uniform(0, 1)          # Mix between L1 and L2 (0=Ridge, 1=Lasso)
}

# Create base model
# Note: For classification with ElasticNet, we need to use:
    # from sklearn.linear_model import SGDClassifier with loss='log_loss' and penalty='elasticnet'
    # here the loss function is the negative log likelihood which in LR is mathematically equivalent to the binary cross-entropy loss
    # OR use LogisticRegression with penalty='elasticnet' (requires solver='saga')
    # note: the optimizer is NOT ILRS (like for normal LR), but rather SGD. we can't use IRLS because the L1 penalty is not differentiable using that.

base_model = SGDClassifier(
    loss='log_loss',  # Logistic regression
    penalty='elasticnet',
    class_weight={0:1, 1:0.8},
    max_iter=10000,
    random_state=seed
)
    
# Create RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dict,
    n_iter=100,  # Number of parameter combinations to try
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
    # StratifiedKFold accounts for possible class imbalance, which regular KFold() doesn't do
    scoring='neg_log_loss',
    # optimize using binary cross entropy loss directly instead of AUC so we don't strictly need to calibrate after.
    random_state=seed,
    n_jobs=-1,  # Use all available cores
    verbose=0
)
    
# Fit random search
random_search.fit(X_train_efi_scaled, y_train_efi)
# random_search now contains:
    # All models trained during CV
    # The best parameters found
    # The best model retrained on ALL training data
    
# Get best model
best_model = random_search.best_estimator_ # this calls up the best model retrained on ALL training data
best_params = random_search.best_params_
best_score = random_search.best_score_

print(f"Best parameters: {best_params}")
print(f"Best CV Log Loss: {-best_score:.3f}")  # best_score is negative log loss, so negate it for positive value

# Calculate max safe folds based on minority class
n_positives = sum(y_train_efi == 1)
n_negatives = sum(y_train_efi == 0)
max_safe_folds = min(n_positives, n_negatives)

# Use reasonable maximum (10 is typical) but don't exceed data limits
n_folds = min(5, max_safe_folds)

print(f"Positive cases: {n_positives}, Negative cases: {n_negatives}")
print(f"Using {n_folds} folds for cross-validation")

# Tune the decision threshold on the best model
# get probabilities for 30 different CV runs of the same (best) model
y_proba_thresh = cross_val_predict(
    best_model, X_train_efi_scaled, y_train_efi, 
    method='predict_proba', cv=StratifiedKFold(n_folds, shuffle=True, random_state=seed)
)[:, 1] 

# Threshold tuning
# use the 30 probabilities to compute a bunch of F1 scores
# identify the best f1 score and designate it as the best cutoff
precision, recall, thresh = precision_recall_curve(y_train_efi, y_proba_thresh)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9) 
optimal_threshold = thresh[np.argmax(f1_scores)] # argmax() gives the index position, not the scalar f1 value, of the highest f1 score

# store into list
for t, f1 in zip(thresh, f1_scores):
    threshold_list_row = {
        'algorithm': 'Elastic Net',
        'EFI': True,
        'threshold': t,
        'f1_scores': f1
    }
    threshold_list_small.append(threshold_list_row)

# EVALUATE ON TEST SET
# Get predictions
y_pred_proba = best_model.predict_proba(X_test_efi_scaled)[:, 1]  # Probabilities for class 1 (dead at 3m), now works with probability=True
y_pred = (y_pred_proba >= optimal_threshold).astype(int)  # Class predictions (0/1)

# Calculate metrics
test_log_loss = log_loss(y_test_efi, y_pred_proba)
f1 = f1_score(y_test_efi, y_pred)
# Get ROC curve data
roc_auc = roc_auc_score(y_test_efi, y_pred_proba)
fpr, tpr, thresholds = roc_curve(y_test_efi, y_pred_proba)
# Get precision-recall curve data
precision, recall, pr_thresholds = precision_recall_curve(y_test_efi, y_pred_proba)
pr_auc = average_precision_score(y_test_efi, y_pred_proba) 
# Get data for calibration analyses
prob_true, prob_pred = calibration_curve(y_test_efi, y_pred_proba, n_bins=5, strategy='uniform')
final_citl, final_slope = bootstrap_calibration_metrics(y_test_efi, y_pred_proba)

print(f"Test Log Loss: {test_log_loss:.3f}")
print(f"Test F1: {f1:.3f}")
print(f"Optimal Threshold: {optimal_threshold:.3f}")
print(f"Test ROC-AUC: {roc_auc:.3f}")
print(f"Test PR-AUC: {pr_auc:.3f}")
print(f"Test Calibration CITL: {final_citl}")
print(f"Test Calibration Slope: {final_slope}")

# Store results
results_table_small.append({
    'algorithm': 'Elastic Net',
    'EFI': True,
    'log_loss': test_log_loss,
    'f1': f1,
    'optimal_threshold': optimal_threshold,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'final_citl': final_citl,
    'final_slope': final_slope
})

# Append to storage list for plotting graph
results_graph_small.append({
    'algorithm': 'Elastic Net',
    'EFI': True,
    'roc_auc': roc_auc,
    'fpr': fpr,
    'tpr': tpr,
    'pr_auc': pr_auc,
    'precision': precision,
    'recall': recall,
    'prob_true': prob_true,
    'prob_pred': prob_pred
})

# Append to pred and proba dicts for running delong and mcnemar tests
models_pred_small['EFI'] = y_pred
models_proba_small['EFI'] = y_pred_proba

In [ ]:
# ===========================
# Learning Curve for Log Loss
# ===========================

from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np

# Generate learning curve with log loss
train_sizes, train_scores, val_scores = learning_curve(
    best_model, 
    X_train_efi_scaled, 
    y_train_efi,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
    scoring='neg_log_loss',  # Note: returns NEGATIVE log loss
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

# Convert back to positive log loss (sklearn returns negative for maximization)
train_log_loss = -train_scores  # Now positive (lower is better)
val_log_loss = -val_scores

# Calculate means
train_mean = np.mean(train_log_loss, axis=1)
train_std = np.std(train_log_loss, axis=1)
val_mean = np.mean(val_log_loss, axis=1)
val_std = np.std(val_log_loss, axis=1)

# Plot (lower log loss is better)
plt.figure(figsize=(8, 6))

plt.plot(train_sizes, train_mean, 'o-', label='Training', color='blue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                 alpha=0.1, color='blue')

plt.plot(train_sizes, val_mean, 'o-', label='Cross-validation', color='red')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                 alpha=0.1, color='red')

plt.xlabel('Number of training examples')
plt.ylabel('Log Loss (lower is better)')
plt.title(f'Learning Curve - Log Loss')
plt.legend(loc='upper right')  # Upper right because lower is better
plt.grid(True, alpha=0.3)

# Diagnostic with Log Loss
final_train_loss = train_mean[-1]
final_val_loss = val_mean[-1]
gap = final_train_loss - final_val_loss  # Train MINUS Val

print(f"Final Train Log Loss: {final_train_loss:.3f}")
print(f"Final Validation Log Loss: {final_val_loss:.3f}")
print(f"Gap (Train - Val): {gap:.3f}")

### NO EFI IN FEATURE SET

In [ ]:
from sklearn.preprocessing import StandardScaler

# Prepare data
X_train_noefi = train_df_small.drop(['id','bi0.25','bi1','death1','death0.25','efi_score'], axis=1)
y_train_noefi = train_df_small['death1']

X_test_noefi = test_df_small.drop(['id','bi0.25','bi1','death1','death0.25','efi_score'], axis=1)
y_test_noefi = test_df_small['death1']
    
# Standardize
scaler = StandardScaler()
X_train_noefi_scaled = scaler.fit_transform(X_train_noefi)
X_test_noefi_scaled = scaler.transform(X_test_noefi)

# inspect
print(f'Train: {X_train_noefi_scaled}')
print(f'Test: {X_test_noefi_scaled}')

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, precision_recall_curve, average_precision_score, log_loss 
from scipy.stats import uniform, loguniform
from sklearn.calibration import calibration_curve


# HYPERPARAMETER TUNING WITH RANDOMIZED SEARCH CV
print("Tuning hyperparameters with 10-fold CV...")

# Define parameter distribution for ElasticNet
param_dict = {
    'alpha': loguniform(0.08, 1.0),  # Regularization strength
    'l1_ratio': uniform(0, 1)          # Mix between L1 and L2 (0=Ridge, 1=Lasso)
}

# Create base model
# Note: For classification with ElasticNet, we need to use:
    # from sklearn.linear_model import SGDClassifier with loss='log_loss' and penalty='elasticnet'
    # here the loss function is the negative log likelihood which in LR is mathematically equivalent to the binary cross-entropy loss
    # OR use LogisticRegression with penalty='elasticnet' (requires solver='saga')
    # note: the optimizer is NOT ILRS (like for normal LR), but rather SGD. we can't use IRLS because the L1 penalty is not differentiable using that.

base_model = SGDClassifier(
    loss='log_loss',  # Logistic regression
    penalty='elasticnet',
    class_weight={0:1, 1:0.8},
    max_iter=10000,
    random_state=seed
)
    
# Create RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dict,
    n_iter=100,  # Number of parameter combinations to try
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
    # StratifiedKFold accounts for possible class imbalance, which regular KFold() doesn't do
    scoring='neg_log_loss',
    # optimize using binary cross entropy loss directly instead of AUC so we don't strictly need to calibrate after.
    random_state=seed,
    n_jobs=-1,  # Use all available cores
    verbose=0
)
    
# Fit random search
random_search.fit(X_train_noefi_scaled, y_train_noefi)
# random_search now contains:
    # All models trained during CV
    # The best parameters found
    # The best model retrained on ALL training data
    
# Get best model
best_model = random_search.best_estimator_ # this calls up the best model retrained on ALL training data
best_params = random_search.best_params_
best_score = random_search.best_score_

print(f"Best parameters: {best_params}")
print(f"Best CV Log Loss: {-best_score:.3f}")  # best_score is negative log loss, so negate it for positive value

# Calculate max safe folds based on minority class
n_positives = sum(y_train_noefi == 1)
n_negatives = sum(y_train_noefi == 0)
max_safe_folds = min(n_positives, n_negatives)

# Use reasonable maximum (10 is typical) but don't exceed data limits
n_folds = min(5, max_safe_folds)

print(f"Positive cases: {n_positives}, Negative cases: {n_negatives}")
print(f"Using {n_folds} folds for cross-validation")

# Tune the decision threshold on the best model
# get probabilities for 30 different CV runs of the same (best) model
y_proba_thresh = cross_val_predict(
    best_model, X_train_noefi_scaled, y_train_noefi, 
    method='predict_proba', cv=StratifiedKFold(n_folds, shuffle=True, random_state=seed)
)[:, 1] 

# Threshold tuning
# use the 30 probabilities to compute a bunch of F1 scores
# identify the best f1 score and designate it as the best cutoff
precision, recall, thresh = precision_recall_curve(y_train_noefi, y_proba_thresh)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9) 
optimal_threshold = thresh[np.argmax(f1_scores)] # argmax() gives the index position, not the scalar f1 value, of the highest f1 score

# store into list
for t, f1 in zip(thresh, f1_scores):
    threshold_list_row = {
        'algorithm': 'Support Vector Machine',
        'EFI': False,
        'threshold': t,
        'f1_scores': f1
    }
    threshold_list_small.append(threshold_list_row)

# EVALUATE ON TEST SET
# Get predictions
y_pred_proba = best_model.predict_proba(X_test_noefi_scaled)[:, 1]  # Probabilities for class 1 (dead at 3m), now works with probability=True
y_pred = (y_pred_proba >= optimal_threshold).astype(int)  # Class predictions (0/1)

# Calculate metrics
test_log_loss = log_loss(y_test_noefi, y_pred_proba)
f1 = f1_score(y_test_noefi, y_pred)
# Get ROC curve data
roc_auc = roc_auc_score(y_test_noefi, y_pred_proba)
fpr, tpr, thresholds = roc_curve(y_test_noefi, y_pred_proba)
# Get precision-recall curve data
precision, recall, pr_thresholds = precision_recall_curve(y_test_noefi, y_pred_proba)
pr_auc = average_precision_score(y_test_noefi, y_pred_proba) 
# Get data for calibration analyses
prob_true, prob_pred = calibration_curve(y_test_noefi, y_pred_proba, n_bins=5, strategy='uniform')
final_citl, final_slope = bootstrap_calibration_metrics(y_test_noefi, y_pred_proba)

print(f"Test Log Loss: {test_log_loss:.3f}")
print(f"Test F1: {f1:.3f}")
print(f"Optimal Threshold: {optimal_threshold:.3f}")
print(f"Test ROC-AUC: {roc_auc:.3f}")
print(f"Test PR-AUC: {pr_auc:.3f}")
print(f"Test Calibration CITL: {final_citl}")
print(f"Test Calibration Slope: {final_slope}")

# Store results
results_table_small.append({
    'algorithm': 'Elastic Net',
    'EFI': False,
    'log_loss': test_log_loss,
    'f1': f1,
    'optimal_threshold': optimal_threshold,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'final_citl': final_citl,
    'final_slope': final_slope
})

# Append to storage list for plotting graph
results_graph_small.append({
    'algorithm': 'Elastic Net',
    'EFI': False,
    'roc_auc': roc_auc,
    'fpr': fpr,
    'tpr': tpr,
    'pr_auc': pr_auc,
    'precision': precision,
    'recall': recall,
    'prob_true': prob_true,
    'prob_pred': prob_pred
})

# Append to pred and proba dicts for running delong and mcnemar tests
models_pred_small['No EFI'] = y_pred
models_proba_small['No EFI'] = y_pred_proba

In [ ]:
# ===========================
# Learning Curve for Log Loss
# ===========================

from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np

# Generate learning curve with log loss
train_sizes, train_scores, val_scores = learning_curve(
    best_model, 
    X_train_noefi_scaled, 
    y_train_noefi,
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
    scoring='neg_log_loss',  # Note: returns NEGATIVE log loss
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

# Convert back to positive log loss (sklearn returns negative for maximization)
train_log_loss = -train_scores  # Now positive (lower is better)
val_log_loss = -val_scores

# Calculate means
train_mean = np.mean(train_log_loss, axis=1)
train_std = np.std(train_log_loss, axis=1)
val_mean = np.mean(val_log_loss, axis=1)
val_std = np.std(val_log_loss, axis=1)

# Plot (lower log loss is better)
plt.figure(figsize=(8, 6))

plt.plot(train_sizes, train_mean, 'o-', label='Training', color='blue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                 alpha=0.1, color='blue')

plt.plot(train_sizes, val_mean, 'o-', label='Cross-validation', color='red')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                 alpha=0.1, color='red')

plt.xlabel('Number of training examples')
plt.ylabel('Log Loss (lower is better)')
plt.title(f'Learning Curve - Log Loss')
plt.legend(loc='upper right')  # Upper right because lower is better
plt.grid(True, alpha=0.3)

# Diagnostic with Log Loss
final_train_loss = train_mean[-1]
final_val_loss = val_mean[-1]
gap = final_train_loss - final_val_loss  # Train MINUS Val

print(f"Final Train Log Loss: {final_train_loss:.3f}")
print(f"Final Validation Log Loss: {final_val_loss:.3f}")
print(f"Gap (Train - Val): {gap:.3f}")

In [ ]:
# generate and see the result tables
results_table_small_df = pd.DataFrame(results_table_small)
results_table_small_df

In [ ]:
# subset
results_table_small_df = results_table_small_df.drop_duplicates(subset='EFI',keep='last').reset_index(drop=True)
results_table_small_df

In [ ]:
# generate and see the graph tables
results_graph_small_df = pd.DataFrame(results_graph_small)
results_graph_small_df

In [ ]:
results_graph_small_df = results_graph_small_df.drop_duplicates(subset='EFI',keep='last').reset_index(drop=True)
results_graph_small_df

In [ ]:
# ===========
# plot graphs 
# ===========

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

# Plot both
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# set colors
colors = plt.cm.tab10(np.linspace(0, 1, len(results_graph_small_df)))
# plt.cm.tab10 is a colormap with 10 distinct colors (good for categorical data)
    # Colors in colormaps are indexed by floats between 0 and 1, not integers
    # plt.cm.tab10 is a "Qualitative" colormap that INTERPOLATES between colors. It creates a BLEND between the two nearest colors

# ROC Curve
for idx, row in results_graph_small_df.iterrows():
    axes[0].plot(row['fpr'], row['tpr'], 
                 color=colors[idx], 
                 label=f"{row['EFI']} (ROC-AUC = {row['roc_auc']:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--') # Plots the diagonal line from (0,0) to (1,1) in black dashed style
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3) # adds a semi-transparent grid

# Calibration Tests
for idx, row in results_graph_small_df.iterrows():
    axes[1].plot(row['prob_pred'], row['prob_true'], 
                 color=colors[idx], 
                 label=row['EFI'])

axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Actual Fraction of Positives')
axes[1].set_title('Calibration Plot')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Add an overall figure title
fig.suptitle('Predictive Performance on Test Data', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# compare models statistically
# ============================
model_names = ['EFI', 'No EFI']

# Run comparisons with Bonferroni correction
results = compare_all_models(
    y_test_efi,
    models_proba_small,
    models_pred_small,
    model_names,
    correction_method=None,  # Options: 'bonferroni', 'holm', 'fdr_bh', None
    alpha=0.05
)

# Access dataframes
auc_comparisons = results['auc_comparisons']
f1_comparisons = results['f1_comparisons']

# inspect
print(f'AUC:\n {auc_comparisons}')
print(f'F1:\n {f1_comparisons}')